In [ ]:
# Vectorized InterAug: equality check + speed benchmark (no training).
# Settings: Accelerator = None (CPU is enough; GPU optional), Internet = On. Run All.
import subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/hada-autodial-zhou-mi-preprocessing'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/tmp/repo')
if (REPO_PATH / '.git').is_dir():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_PATH)], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH, check=True)
sys.path.insert(0, str(REPO_PATH))

import torch
from utils.interaug import interaug, interaug_vectorized

# (name, batch, channels, samples, classes) as produced by each datamodule.
CASES = [
    ('BCI IV-2a', 48, 22, 1000, 4),
    ('BCI IV-2b', 48, 3, 1000, 2),
    ('Zhou2016', 32, 14, 1125, 3),
    ('PhysioNet BS=108', 108, 64, 656, 4),
    ('PhysioNet BS=144', 144, 64, 656, 4),
]
devices = ['cpu'] + (['cuda'] if torch.cuda.is_available() else [])

# ---- 1. Same seed -> identical output ----
for name, n, c, t, k in CASES:
    for device in devices:
        for seed in range(5):
            g = torch.Generator().manual_seed(1000 + seed)
            x = torch.randn(n, c, t, generator=g).to(device)
            y = torch.randint(0, k, (n,), generator=g).to(device)
            if seed == 4:
                y[: n // 2] = 0   # unbalanced batch
            torch.manual_seed(seed); xa, ya = interaug([x.clone(), y.clone()])
            torch.manual_seed(seed); xb, yb = interaug_vectorized([x.clone(), y.clone()])
            assert torch.equal(xa, xb) and torch.equal(ya, yb), (name, device, seed)
    print(f'identical  {name:18s} devices={devices}')
print('All outputs identical to the original InterAug.\n')

# ---- 2. Speed, one CPU thread as inside a DataLoader worker ----
torch.set_num_threads(1)
print(f"{'case':18s} {'loop ms':>9s} {'vector ms':>10s} {'speed-up':>9s}")
for name, n, c, t, k in CASES:
    x = torch.randn(n, c, t); y = torch.randint(0, k, (n,))
    timings = {}
    for label, fn in (('loop', interaug), ('vector', interaug_vectorized)):
        fn([x, y])
        start = time.perf_counter(); reps = 20
        for _ in range(reps):
            fn([x, y])
        timings[label] = (time.perf_counter() - start) / reps * 1000
    print(f"{name:18s} {timings['loop']:9.1f} {timings['vector']:10.1f} {timings['loop'] / timings['vector']:8.1f}x")

# ---- 3. What it means for one PhysioNet epoch (~9,000 source trials) ----
for bs in (108, 144):
    x = torch.randn(bs, 64, 656); y = torch.randint(0, 4, (bs,))
    per_batch = {}
    for label, fn in (('loop', interaug), ('vector', interaug_vectorized)):
        start = time.perf_counter()
        for _ in range(10):
            fn([x, y])
        per_batch[label] = (time.perf_counter() - start) / 10
    batches = 9000 // bs
    print(f'PhysioNet BS={bs}: InterAug CPU time per epoch, single thread: '
          f"loop {per_batch['loop'] * batches:.0f} s, vectorized {per_batch['vector'] * batches:.0f} s")
